# PharmVigiNet — Colab Training

Trains PubMedBERT text model and ChemBERTa mol model on FAERS data.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 or A100)
2. In Google Drive, create a folder named `pharmviginet` with this structure:
   ```
   pharmviginet/
   ├── ml/
   │   ├── train.parquet
   │   ├── val.parquet
   │   └── test.parquet
   └── drug_smiles_map.parquet
   ```
3. Right-click the `pharmviginet` folder → Share → Anyone with the link → Viewer
4. Paste the folder share link in cell 1 below

**Folder share link looks like:**
```
https://drive.google.com/drive/folders/FOLDER_ID_HERE?usp=sharing
```

In [ ]:
# ── 1. Download data from shared Drive folder ─────────────────────────────────
# Paste your folder share link here (folder must be shared as "Anyone with the link")
FOLDER_URL = 'https://drive.google.com/drive/folders/1phzT5AC2qP9y1_FDdwBuw-vB3J6s8a1l?usp=sharing'

import subprocess, shutil
from pathlib import Path

subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
import gdown

DATA_DIR = Path('/content/pharmviginet')

if DATA_DIR.exists() and any(DATA_DIR.rglob('*.parquet')):
    print('Data already present, skipping download.')
else:
    print('Downloading folder from Google Drive ...')
    gdown.download_folder(url=FOLDER_URL, output='/content/', quiet=False, use_cookies=False)

    # Normalize folder name — gdown uses the Drive folder name which may differ
    if not DATA_DIR.exists():
        candidates = [p for p in Path('/content').iterdir()
                      if p.is_dir() and (p / 'ml').exists()]
        if not candidates:
            raise RuntimeError('Download failed — no folder with ml/ found under /content/')
        shutil.move(str(candidates[0]), str(DATA_DIR))
        print(f'Renamed {candidates[0].name} → pharmviginet')

# Verify expected files
expected = [
    DATA_DIR / 'ml/train.parquet',
    DATA_DIR / 'ml/val.parquet',
    DATA_DIR / 'ml/test.parquet',
    DATA_DIR / 'drug_smiles_map.parquet',
]
all_ok = True
for p in expected:
    if p.exists():
        print(f'  {p.relative_to(DATA_DIR)}: {p.stat().st_size/1e9:.2f} GB')
    else:
        print(f'  MISSING: {p.relative_to(DATA_DIR)} — check folder structure')
        all_ok = False

if all_ok:
    print('All files ready.')

In [ ]:
# ── 2. Install deps ───────────────────────────────────────────────────────────
!pip install -q transformers==4.40.0 pyarrow scikit-learn

In [ ]:
# ── 3. Paths ──────────────────────────────────────────────────────────────────
import os
from pathlib import Path

DATA_DIR   = Path('/content/pharmviginet')
TRAIN_PATH = DATA_DIR / 'ml/train.parquet'
VAL_PATH   = DATA_DIR / 'ml/val.parquet'
TEST_PATH  = DATA_DIR / 'ml/test.parquet'
CKPT_DIR   = DATA_DIR / 'checkpoints'
HF_CACHE   = DATA_DIR / 'hf_cache'

CKPT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE)

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    size = f'{p.stat().st_size/1e9:.1f} GB' if p.exists() else 'MISSING — re-run cell 1'
    print(f'  {p.name}: {size}')

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn

PUBMEDBERT   = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
CHEMBERT     = 'seyonec/ChemBERTa-zinc-base-v2'
TEXT_MAX_LEN = 128
MOL_MAX_LEN  = 128
BATCH_SIZE   = 32   # per GPU — effective batch = BATCH_SIZE * n_gpus
LR           = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS   = 3
POS_WEIGHT   = 15.0
TRAIN_SAMPLE = 500_000   # rows — increase to None for full dataset
VAL_SAMPLE   = 50_000

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS  = torch.cuda.device_count()
print(f'Device: {DEVICE} | GPUs: {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ── 5. Dataset ────────────────────────────────────────────────────────────────
import random
import pyarrow.parquet as pq
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class FAERSDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=TEXT_MAX_LEN, sample_n=None, seed=42):
        print(f'Loading {Path(path).name} ...')
        tbl = pq.read_table(path, columns=['text_input', 'smiles', 'label'])
        texts  = tbl['text_input'].to_pylist()
        smiles = tbl['smiles'].to_pylist()
        labels = tbl['label'].to_pylist()
        if sample_n and sample_n < len(texts):
            rng = random.Random(seed)
            idx = rng.sample(range(len(texts)), sample_n)
            texts  = [texts[i]  for i in idx]
            smiles = [smiles[i] for i in idx]
            labels = [labels[i] for i in idx]
        self.texts, self.smiles, self.labels = texts, smiles, labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        print(f'  {len(self.texts):,} rows, {np.mean([int(l or 0) for l in labels]):.1%} positive')

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx] or '',
            truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'smiles':         self.smiles[idx] or '[UNK-MOL]',
            'label': torch.tensor(int(self.labels[idx] or 0), dtype=torch.float),
        }

print('Dataset class ready.')

In [ ]:
# ── 6. Text Model (PubMedBERT) ────────────────────────────────────────────────
import torch.nn as nn
from transformers import AutoModel

class TextModel(nn.Module):
    def __init__(self, model_name=PUBMEDBERT, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.bert.config.hidden_size, 1),
        )

    def forward(self, input_ids, attention_mask):
        cls = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.classifier(cls).squeeze(-1)

print('TextModel class ready.')

In [ ]:
# ── 7. Metrics helpers ────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

def compute_metrics(y_true, y_score):
    mask = np.isfinite(y_score)
    y_true, y_score = np.array(y_true)[mask], np.array(y_score)[mask]
    return {
        'auc':   roc_auc_score(y_true, y_score),
        'auprc': average_precision_score(y_true, y_score),
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    scores, labels = [], []
    for batch in loader:
        logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        scores.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        labels.extend(batch['label'].numpy().tolist())
    return compute_metrics(labels, scores)

print('Metrics helpers ready.')

In [ ]:
# ── 8. Train text model ───────────────────────────────────────────────────────
import time, json
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(PUBMEDBERT)

train_ds = FAERSDataset(TRAIN_PATH, tokenizer, sample_n=TRAIN_SAMPLE)
val_ds   = FAERSDataset(VAL_PATH,   tokenizer, sample_n=VAL_SAMPLE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE * max(N_GPUS, 1), shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * max(N_GPUS, 1) * 2, shuffle=False, num_workers=2, pin_memory=True)

model = TextModel().to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'Using DataParallel across {N_GPUS} GPUs')

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, total_steps//10, total_steps)

best_auc, history = 0.0, []
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss, n, t0 = 0.0, 0, time.time()
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        loss = criterion(logits, batch['label'].to(DEVICE))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item() * len(batch['label']); n += len(batch['label'])
        if (i+1) % 500 == 0:
            print(f'  step {i+1}/{len(train_loader)} loss={total_loss/n:.4f}')

    metrics = evaluate(model, val_loader)
    elapsed = time.time() - t0
    print(f'Epoch {epoch}/{MAX_EPOCHS} | loss={total_loss/n:.4f} | '
          f'AUC={metrics["auc"]:.4f} AUPRC={metrics["auprc"]:.4f} | {elapsed/60:.1f}min')
    history.append({'epoch': epoch, 'loss': total_loss/n, **metrics})

    if metrics['auc'] > best_auc:
        best_auc = metrics['auc']
        # Save unwrapped model state (works for both DataParallel and single GPU)
        state = model.module.state_dict() if N_GPUS > 1 else model.state_dict()
        torch.save(state, CKPT_DIR / 'text_best.pt')
        print(f'  → New best AUC {best_auc:.4f} saved')

with open(CKPT_DIR / 'text_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'\nBest val AUC: {best_auc:.4f}')
print(f'Baseline to beat (ROR_train): 0.8928')

In [ ]:
# ── 9. Evaluate best model on test set ───────────────────────────────────────
print('Loading best checkpoint ...')
model.load_state_dict(torch.load(CKPT_DIR / 'text_best.pt', map_location=DEVICE))

test_ds     = FAERSDataset(TEST_PATH, tokenizer, sample_n=100_000)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)
test_metrics = evaluate(model, test_loader)
print(f'Test AUC={test_metrics["auc"]:.4f} AUPRC={test_metrics["auprc"]:.4f}')
print(f'ROR_train baseline: AUC=0.8928')
print(f'Delta: {test_metrics["auc"]-0.8928:+.4f}')

## Next: Mol Model (ChemBERTa)

Run the cell below to train ChemBERTa on SMILES strings.

In [ ]:
# ── 10. Mol Model (ChemBERTa on SMILES) ──────────────────────────────────────

class MolDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=MOL_MAX_LEN, sample_n=None, seed=42):
        print(f'Loading {Path(path).name} for mol ...')
        tbl = pq.read_table(path, columns=['smiles', 'label'])
        smiles = tbl['smiles'].to_pylist()
        labels = tbl['label'].to_pylist()
        if sample_n and sample_n < len(smiles):
            rng = random.Random(seed)
            idx = rng.sample(range(len(smiles)), sample_n)
            smiles = [smiles[i] for i in idx]
            labels = [labels[i] for i in idx]
        self.smiles, self.labels = smiles, labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        print(f'  {len(self.smiles):,} rows')

    def __len__(self): return len(self.smiles)

    def __getitem__(self, idx):
        smi = self.smiles[idx] or '[UNK-MOL]'
        enc = self.tokenizer(
            smi, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(int(self.labels[idx] or 0), dtype=torch.float),
        }

mol_tokenizer    = AutoTokenizer.from_pretrained(CHEMBERT)
mol_train_ds     = MolDataset(TRAIN_PATH, mol_tokenizer, sample_n=TRAIN_SAMPLE)
mol_val_ds       = MolDataset(VAL_PATH,   mol_tokenizer, sample_n=VAL_SAMPLE)
mol_train_loader = DataLoader(mol_train_ds, batch_size=BATCH_SIZE * max(N_GPUS, 1), shuffle=True,  num_workers=2, pin_memory=True)
mol_val_loader   = DataLoader(mol_val_ds,   batch_size=BATCH_SIZE * max(N_GPUS, 1) * 2, shuffle=False, num_workers=2)

mol_model = TextModel(model_name=CHEMBERT).to(DEVICE)
if N_GPUS > 1:
    mol_model = nn.DataParallel(mol_model)
    print(f'Using DataParallel across {N_GPUS} GPUs')

mol_optimizer = AdamW(mol_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
mol_total     = len(mol_train_loader) * MAX_EPOCHS
mol_scheduler = get_linear_schedule_with_warmup(mol_optimizer, mol_total//10, mol_total)
mol_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))

@torch.no_grad()
def eval_mol(model, loader):
    model.eval()
    scores, labels = [], []
    for batch in loader:
        logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        scores.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        labels.extend(batch['label'].numpy().tolist())
    return compute_metrics(labels, scores)

best_mol_auc, mol_history = 0.0, []
for epoch in range(1, MAX_EPOCHS + 1):
    mol_model.train()
    total_loss, n, t0 = 0.0, 0, time.time()
    for i, batch in enumerate(mol_train_loader):
        mol_optimizer.zero_grad()
        logits = mol_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        loss = mol_criterion(logits, batch['label'].to(DEVICE))
        loss.backward()
        nn.utils.clip_grad_norm_(mol_model.parameters(), 1.0)
        mol_optimizer.step(); mol_scheduler.step()
        total_loss += loss.item() * len(batch['label']); n += len(batch['label'])

    metrics = eval_mol(mol_model, mol_val_loader)
    print(f'Mol Epoch {epoch}/{MAX_EPOCHS} | loss={total_loss/n:.4f} | '
          f'AUC={metrics["auc"]:.4f} | {(time.time()-t0)/60:.1f}min')
    mol_history.append({'epoch': epoch, 'loss': total_loss/n, **metrics})
    if metrics['auc'] > best_mol_auc:
        best_mol_auc = metrics['auc']
        state = mol_model.module.state_dict() if N_GPUS > 1 else mol_model.state_dict()
        torch.save(state, CKPT_DIR / 'mol_best.pt')
        print(f'  → New best mol AUC {best_mol_auc:.4f}')

with open(CKPT_DIR / 'mol_history.json', 'w') as f:
    json.dump(mol_history, f, indent=2)
print(f'Best mol val AUC: {best_mol_auc:.4f}')

In [ ]:
# ── 11. Download checkpoints to your machine ──────────────────────────────────
# Run this cell when training is done — downloads all checkpoints + history logs
from google.colab import files

for f in CKPT_DIR.iterdir():
    print(f'Downloading {f.name} ({f.stat().st_size/1e6:.1f} MB) ...')
    files.download(str(f))

print('Done. Save these to model_checkpoints/ in your local repo.')